In [ ]:
!pip install pennylane biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 61.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 21.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 66.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 39.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 62.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 90.5 MB/s  0:00:00
  Attempting uninstall: autograd
    Found existing installation: autograd 1.9.1
    Uninstalling autograd-1.9.1:
      Successfully uninstalled autograd-1.9.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [pennylane]


In [ ]:
#this cell holds the whole ideal-circuit pipeline as functions, so the same steps can be run on any graph.
#the steps are: normalization and Chebyshev polynomials (degree fixed by DELTA) -> the two ideal QSVT circuits -> classical SC reference -> ranking comparison.
#bipartite graphs have no odd closed walks, so for them only the even circuit is run (the sinh part is exactly zero).
#the polynomial degree depends only on beta, the normalization alpha and DELTA. No classical score is computed before the circuits are built.
import numpy as np, networkx as nx, pandas as pd, time, warnings
import pennylane as qml
from scipy.linalg import expm
from scipy.special import ive
from scipy.stats import spearmanr, rankdata
from numpy.polynomial import chebyshev as C
warnings.filterwarnings("ignore")

DELTA = 1e-4   # target absolute error on the SC scale; SC(v) >= 1 for unweighted graphs, so it also bounds the relative error. One fixed value for all graphs.

def tie_classes(sc, tol=1e-9):                    # vertices whose SC agree to 1e-9 (relative) are one class
    order = np.argsort(-sc)
    cls = np.zeros(len(sc), dtype=int)
    for a, b in zip(order[:-1], order[1:]):
        cls[b] = cls[a] + int(abs(sc[a] - sc[b]) > tol * abs(sc[a]))
    return cls                                    # class 0 = most central

def cheb_coeffs(z, deg, parity):
    k = np.arange(deg + 1)
    c = 2.0 * ive(k, z)
    c[0] = ive(0, z)
    c[k % 2 != parity] = 0.0
    if parity == 0:
        return c / (0.5 * (1 + np.exp(-2 * z)))    # divide by cosh(z), scaled by e^-z to match ive
    return c / (0.5 * (1 - np.exp(-2 * z)))        # divide by sinh(z), scaled by e^-z to match ive

def tail_error(z, deg, parity):
    return float(np.sum(np.abs(cheb_coeffs(z, deg + 60, parity)[deg + 1:])))

def degree_for_eps(z, eps, parity, max_deg=300):
    d = parity
    while tail_error(z, d, parity) > eps:
        d += 2
        if d > max_deg:
            raise RuntimeError("tolerance not reached; raise max_deg")
    return d, cheb_coeffs(z, d, parity)

def compare(sc_exact, sc_est, k, tol=1e-9):
    # The circuit scores are used as they are. Vertices that are tied in the classical SC may come out in either order.
    ex = np.asarray(sc_exact, dtype=float); est = np.asarray(sc_est, dtype=float); n = len(ex)
    cls = tie_classes(ex, tol)
    # tau: Kendall correlation over the pairs the classical SC orders strictly (= 2 * fraction of correctly ordered pairs - 1)
    iu = np.triu_indices(n, 1)
    dex = (ex[:, None] - ex[None, :])[iu]; dest = (est[:, None] - est[None, :])[iu]
    strict = np.abs(dex) > tol * np.maximum(np.abs(ex[:, None]), np.abs(ex[None, :]))[iu]
    tau = 2 * float(np.mean(np.sign(dex[strict]) == np.sign(dest[strict]))) - 1 if strict.any() else np.nan
    # rho: Spearman correlation of ranks; inside a classical tie class the ranks (classical and estimated) are replaced by their mean rank,
    # so the result does not depend on the order within the class (the scores themselves are not changed)
    r_ex = rankdata(-ex); r_est = rankdata(-est)
    for c in np.unique(cls):
        r_ex[cls == c] = r_ex[cls == c].mean(); r_est[cls == c] = r_est[cls == c].mean()
    rho = spearmanr(r_ex, r_est)[0] if cls.max() > 0 else np.nan
    # top-k: vertices strictly above the k-th classical value must be in the circuit's top-k; ties at the cut count in either order
    t = np.sort(ex)[::-1][k - 1]
    must = set(np.where(ex > t * (1 + tol))[0]); opt = set(np.where(np.abs(ex - t) <= tol * t)[0])
    top = set(np.argsort(-est, kind="stable")[:k])
    overlap = (len(top & must) + min(len(top & opt), k - len(must))) / k
    return dict(rho=rho, tau=tau, overlap=overlap)

def run_graph(G, beta=1.0, weighted=False, delta=None):
    t0 = time.time()
    delta = DELTA if delta is None else delta
    nodes = sorted(G.nodes()); n = len(nodes)
    q = int(np.ceil(np.log2(n))); N = 2 ** q                       # padding: add isolated dummy vertices up to 2^q
    A = nx.to_numpy_array(G, nodelist=nodes, weight="weight" if weighted else None)
    A_p = np.zeros((N, N)); A_p[:n, :n] = A
    bip = nx.is_bipartite(G)
    # ---- normalization and polynomials: the degree comes from beta, alpha and DELTA only ----
    lam = np.max(np.abs(np.linalg.eigvalsh(A)))
    alpha = 1.001 * max(lam, np.sqrt(np.max(np.abs(A @ A).sum(axis=1))))
    z = beta * alpha
    d_e, c_e = degree_for_eps(z, delta / np.cosh(z), 0)
    d_o, c_o = (None, None) if bip else degree_for_eps(z, delta / np.sinh(z), 1)
    # ---- the ideal circuits ----
    At = A_p / alpha
    be_wires = range(q + 1); rp = q + 1
    BE = qml.BlockEncode(At, wires=be_wires)
    dev = qml.device("default.qubit", wires=q + 2)
    def make_circuit(c):
        angles = qml.poly_to_angles(C.cheb2poly(c), "QSVT")
        pos = [qml.PCPhase(a,  dim=N, wires=be_wires) for a in angles]
        neg = [qml.PCPhase(-a, dim=N, wires=be_wires) for a in angles]
        @qml.qnode(dev)
        def circ(v):
            qml.BasisState(np.array([int(b) for b in format(v, f"0{q}b")]), wires=range(1, 1 + q))
            qml.Hadamard(wires=rp)
            qml.ctrl(qml.QSVT(BE, pos), control=rp, control_values=[0])
            qml.ctrl(qml.QSVT(BE, neg), control=rp, control_values=[1])
            qml.Hadamard(wires=rp)
            return qml.probs(wires=range(q + 2))
        return circ
    def p_success(circ, v):
        bits = [0] + [int(b) for b in format(v, f"0{q}b")] + [0]
        return circ(v)[int("".join(map(str, bits)), 2)]
    circ_e = make_circuit(c_e)
    p_e = np.array([p_success(circ_e, v) for v in range(n)])
    if bip:
        p_o = np.zeros(n)
    else:
        circ_o = make_circuit(c_o)
        p_o = np.array([p_success(circ_o, v) for v in range(n)])
    SC_qsvt = np.cosh(z) * np.sqrt(p_e) + np.sinh(z) * np.sqrt(p_o)
    # ---- classical reference (numpy and networkx), used only to check the circuit ----
    SC_exact = np.diag(expm(beta * A))
    nx_diff = np.nan
    if beta == 1.0 and not weighted:                              # networkx's subgraph_centrality is unweighted
        sc_nx = nx.subgraph_centrality(G)
        nx_diff = max(abs(sc_nx[v] - SC_exact[i]) for i, v in enumerate(nodes))
    # ---- comparison ----
    k = 3 if n <= 8 else 5
    m = compare(SC_exact, SC_qsvt, k)
    return dict(n=n, edges=G.number_of_edges(), qubits=q + 2, bipartite=bip, classes=int(tie_classes(SC_exact).max() + 1),
                alpha=alpha, deg_even=d_e, deg_odd=d_o, pe_min=p_e.min(), po_min=(np.nan if bip else p_o.min()),
                nx_diff=nx_diff, MAE=np.mean(np.abs(SC_qsvt - SC_exact)), max_rel_err=np.max(np.abs(SC_qsvt - SC_exact) / SC_exact),
                rho=m["rho"], tau=m["tau"], topk=k, overlap=m["overlap"], seconds=time.time() - t0)

In [ ]:
# ---- RIN builder following the procedure of the RinQ paper (2025): C-alpha atoms, 8 A cutoff, unweighted ----
from Bio import PDB
from Bio.PDB import PDBList

def build_rinq(pdb_id, cutoff=8.0):
    path = PDBList().retrieve_pdb_file(pdb_id, file_format="pdb", pdir=".", overwrite=False)
    structure = PDB.PDBParser(QUIET=True).get_structure(pdb_id, path)
    G = nx.Graph()
    for chain in structure.get_chains():            # all chains of all models; nodes are identified by residue number
        for ri in chain:
            for rj in chain:
                if ri != rj and "CA" in ri and "CA" in rj:
                    if np.linalg.norm(ri["CA"].coord - rj["CA"].coord) < cutoff:
                        G.add_edge(ri.get_id()[1], rj.get_id()[1])
    return G

# number of residues stated in the RinQ paper, kept for comparison with the built graphs
rinq_residues = {"1XY1": 10, "4D5M": 11, "2N08": 12, "6A5J": 13, "6RQS": 18, "1Q71": 21,
                 "2MLT": 27, "1A7F": 29, "1GCN": 29, "2K6O": 37, "1JL9": 51, "1UBQ": 76}

In [ ]:
#this cell calibrates DELTA once, on the hardest cases: the 12 RIN proteins, the 10 benchmark graphs and the two 8-vertex graphs, all noiseless and at beta = 1.
#every graph is run at DELTA = 1e-1, 1e-2, 1e-3 and 1e-4. A run passes if every pair that the classical SC orders strictly is ordered correctly (tau = 1) and the top-k is valid.
#rule: the loosest DELTA at which every graph passes (also at every tighter value) is found, and the fixed value is one step tighter than that, as a margin.
#results are stored in `calib`, so the cell can be run again and it continues where it stopped.
bip8 = nx.Graph([('u0','v0'),('u0','v1'),('u0','v2'),('u0','v3'),('u1','v0'),('u1','v1'),('u1','v3'),('u2','v1'),('u2','v2'),('u3','v0')])
nonbip8 = nx.Graph([('a','b'),('a','c'),('a','d'),('a','e'),('a','f'),('b','c'),('b','d'),('c','g'),('c','h'),('d','g'),('e','g'),('f','h'),('g','h')])
calib_graphs = {
    "Bull": nx.bull_graph, "K3,5": lambda: nx.complete_bipartite_graph(3, 5), "K5": lambda: nx.complete_graph(5),
    "Sedgewick-Maze": nx.sedgewick_maze_graph, "Florentine": nx.florentine_families_graph,
    "Lollipop L10,20": lambda: nx.lollipop_graph(10, 20), "Davis women": nx.davis_southern_women_graph,
    "Karate club": nx.karate_club_graph, "Tutte": nx.tutte_graph, "BA(50,5,7)": lambda: nx.barabasi_albert_graph(50, 5, seed=7),
    "8-vertex bipartite": lambda: bip8, "8-vertex non-bipartite": lambda: nonbip8,
}
for pid in rinq_residues:
    calib_graphs[pid] = (lambda p=pid: build_rinq(p))

deltas = [1e-1, 1e-2, 1e-3, 1e-4]
try: calib
except NameError: calib = {}
for name, make in calib_graphs.items():
    G = make()
    for d in deltas:
        if (name, d) in calib: continue
        r = run_graph(G, beta=1.0, delta=d)
        ok = bool((np.isnan(r["tau"]) or r["tau"] >= 1 - 1e-12) and r["overlap"] == 1.0)
        calib[(name, d)] = dict(ok=ok, maxrel=r["max_rel_err"], deg=max(r["deg_even"], r["deg_odd"] or 0), sec=r["seconds"])
        print(f"{name:<22} DELTA = {d:.0e}  degree = {calib[(name, d)]['deg']:>2}  pass = {ok!s:<5}  max rel err = {r['max_rel_err']:.1e}  ({r['seconds']:.0f} s)", flush=True)

# ---- summary: pass/fail per graph and DELTA ----
tab = pd.DataFrame({f"{d:.0e}": {n: ("pass" if calib[(n, d)]["ok"] else "FAIL") for n in calib_graphs} for d in deltas})
print("\n" + tab.to_string())
print("\nworst max rel err over all graphs:", {f"{d:.0e}": f"{max(calib[(n, d)]['maxrel'] for n in calib_graphs):.1e}" for d in deltas})
all_pass = [all(calib[(n, d)]["ok"] for n in calib_graphs) for d in deltas]
loosest = next((i for i in range(len(deltas)) if all(all_pass[i:])), None)
if loosest is None:
    print("\nno DELTA in the grid passes on every graph; extend the grid to smaller values")
else:
    frozen = deltas[min(loosest + 1, len(deltas) - 1)]
    print(f"\nloosest DELTA passing on every graph: {deltas[loosest]:.0e}   ->   fixed DELTA (one step tighter): {frozen:.0e}")

Bull                   DELTA = 1e-01  degree =  4  pass = True   max rel err = 1.1e-02  (0 s)
Bull                   DELTA = 1e-02  degree =  6  pass = True   max rel err = 6.8e-04  (1 s)
Bull                   DELTA = 1e-03  degree =  7  pass = True   max rel err = 4.4e-05  (1 s)
Bull                   DELTA = 1e-04  degree =  8  pass = True   max rel err = 6.1e-06  (0 s)
K3,5                   DELTA = 1e-01  degree =  6  pass = True   max rel err = 2.6e-03  (0 s)
K3,5                   DELTA = 1e-02  degree =  8  pass = True   max rel err = 6.1e-05  (0 s)
K3,5                   DELTA = 1e-03  degree =  8  pass = True   max rel err = 6.1e-05  (0 s)
K3,5                   DELTA = 1e-04  degree = 10  pass = True   max rel err = 2.7e-06  (0 s)
K5                     DELTA = 1e-01  degree =  6  pass = True   max rel err = 6.9e-03  (0 s)
K5                     DELTA = 1e-02  degree =  8  pass = True   max rel err = 9.5e-05  (0 s)
K5                     DELTA = 1e-03  degree =  9  pass = Tr

In [ ]:
#this cell tests the fixed DELTA on graphs that were not used to choose it: 30 random graphs (10 Erdos-Renyi, 10 Barabasi-Albert, 10 Watts-Strogatz) from a fixed seed, at beta = 1.
#nothing is tuned here. DELTA is the value set in the functions cell, and every graph is run once. A run passes under the same rule as the calibration (all strictly ordered pairs correct, valid top-k).
rng = np.random.default_rng(2026)
def largest_cc(G):
    H = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    return nx.convert_node_labels_to_integers(H)

held = {}
for i in range(10):
    while True:
        n = int(rng.integers(12, 49)); G = largest_cc(nx.erdos_renyi_graph(n, rng.uniform(3, 6) / n, seed=int(rng.integers(10**6))))
        if G.number_of_nodes() >= 10: break
    held[f"ER-{i}"] = G
for i in range(10):
    n = int(rng.integers(12, 49)); held[f"BA-{i}"] = nx.barabasi_albert_graph(n, int(rng.integers(1, 5)), seed=int(rng.integers(10**6)))
for i in range(10):
    n = int(rng.integers(12, 49)); held[f"WS-{i}"] = nx.connected_watts_strogatz_graph(n, 4, rng.uniform(0.1, 0.5), seed=int(rng.integers(10**6)))

res = {}
for name, G in held.items():
    r = run_graph(G, beta=1.0)
    ok = bool((np.isnan(r["tau"]) or r["tau"] >= 1 - 1e-12) and r["overlap"] == 1.0)
    res[name] = dict(n=r["n"], edges=r["edges"], classes=r["classes"], deg=max(r["deg_even"], r["deg_odd"] or 0), ok=ok,
                     tau=r["tau"], overlap=r["overlap"], maxrel=r["max_rel_err"], sec=r["seconds"])
    print(f"{name:<6} n = {r['n']:>2}  edges = {r['edges']:>3}  degree = {res[name]['deg']:>2}  pass = {ok!s:<5}  max rel err = {r['max_rel_err']:.1e}  ({r['seconds']:.0f} s)", flush=True)

tab = pd.DataFrame(res).T
tab["family"] = [i.split("-")[0] for i in tab.index]
print(f"\nDELTA = {DELTA:.0e}   passed {int(tab['ok'].sum())} of {len(tab)}")
print(tab.groupby("family")["ok"].agg(passed="sum", total="count").to_string())
print("worst max rel err:", f"{tab['maxrel'].astype(float).max():.1e}")
if not tab["ok"].all():
    print("\nfailed graphs:\n", tab[~tab["ok"].astype(bool)][["n", "edges", "classes", "deg", "tau", "overlap"]].to_string())

ER-0   n = 43  edges = 117  degree = 18  pass = True   max rel err = 2.3e-06  (33 s)
ER-1   n = 25  edges =  54  degree = 14  pass = True   max rel err = 4.7e-06  (3 s)
ER-2   n = 35  edges =  89  degree = 17  pass = True   max rel err = 2.3e-06  (14 s)
ER-3   n = 36  edges =  68  degree = 14  pass = True   max rel err = 4.3e-06  (13 s)
ER-4   n = 40  edges =  76  degree = 16  pass = True   max rel err = 2.2e-06  (15 s)
ER-5   n = 18  edges =  62  degree = 18  pass = True   max rel err = 2.5e-07  (2 s)
ER-6   n = 22  edges =  50  degree = 16  pass = True   max rel err = 5.6e-06  (3 s)
ER-7   n = 16  edges =  36  degree = 14  pass = True   max rel err = 1.1e-05  (1 s)
ER-8   n = 35  edges =  68  degree = 13  pass = True   max rel err = 4.7e-06  (12 s)
ER-9   n = 17  edges =  29  degree = 11  pass = True   max rel err = 6.8e-06  (1 s)
BA-0   n = 43  edges = 120  degree = 21  pass = True   max rel err = 2.3e-06  (23 s)
BA-1   n = 27  edges =  50  degree = 14  pass = True   max rel err = 9

In [ ]:
# ---- run all 12 RinQ proteins: ideal QSVT SC (beta = 1) vs classical SC (the Estrada centrality) ----
rows = {}
for pid, n_paper in rinq_residues.items():
    try:
        G = build_rinq(pid)
        r = run_graph(G, beta=1.0, weighted=False)
        m = round(r["overlap"] * r["topk"])
        r["jaccard"] = m / (2 * r["topk"] - m)                  # Jaccard@k from the number of shared top-k vertices
        r["n_paper"] = n_paper
        rows[pid] = r
        print(f"{pid}: n={r['n']} (paper {n_paper}), qubits={r['qubits']}, rho={r['rho']:.3f}, tau={r['tau']:.3f}, "
              f"overlap@{r['topk']}={r['overlap']:.2f}, jaccard={r['jaccard']:.2f}  ({r['seconds']:.0f} s)")
    except Exception as e:
        print(f"{pid}: FAILED -> {type(e).__name__}: {e}")

tab_rin = pd.DataFrame(rows).T[["n_paper", "n", "edges", "qubits", "classes", "deg_even", "deg_odd", "pe_min", "po_min",
                                "nx_diff", "MAE", "max_rel_err", "rho", "tau", "topk", "overlap", "jaccard"]]
fmt = {"pe_min": "{:.1e}".format, "po_min": "{:.1e}".format, "nx_diff": "{:.1e}".format, "MAE": "{:.1e}".format,
       "max_rel_err": "{:.1e}".format, "rho": "{:.3f}".format, "tau": "{:.3f}".format,
       "overlap": "{:.2f}".format, "jaccard": "{:.2f}".format}
print(tab_rin.to_string(formatters=fmt))

Structure exists: './pdb1xy1.ent' 
1XY1: n=9 (paper 10), qubits=6, rho=1.000, tau=1.000, overlap@5=1.00, jaccard=1.00  (1 s)
Structure exists: './pdb4d5m.ent' 
4D5M: n=10 (paper 11), qubits=6, rho=1.000, tau=1.000, overlap@5=1.00, jaccard=1.00  (3 s)
Structure exists: './pdb2n08.ent' 
2N08: n=11 (paper 12), qubits=6, rho=1.000, tau=1.000, overlap@5=1.00, jaccard=1.00  (2 s)
Structure exists: './pdb6a5j.ent' 
6A5J: n=13 (paper 13), qubits=6, rho=1.000, tau=1.000, overlap@5=1.00, jaccard=1.00  (1 s)
Structure exists: './pdb6rqs.ent' 
6RQS: n=16 (paper 18), qubits=6, rho=1.000, tau=1.000, overlap@5=1.00, jaccard=1.00  (1 s)
Structure exists: './pdb1q71.ent' 
1Q71: n=21 (paper 21), qubits=7, rho=1.000, tau=1.000, overlap@5=1.00, jaccard=1.00  (3 s)
Structure exists: './pdb2mlt.ent' 
2MLT: n=26 (paper 27), qubits=7, rho=1.000, tau=1.000, overlap@5=1.00, jaccard=1.00  (3 s)
Structure exists: './pdb1a7f.ent' 
1A7F: n=29 (paper 29), qubits=7, rho=1.000, tau=1.000, overlap@5=1.00, jaccard=1.00 

In [ ]:
# ---- compare graph size with the residues present in each structure file ----
def residue_report(pdb_id):
    st = PDB.PDBParser(QUIET=True).get_structure(pdb_id, f"./pdb{pdb_id.lower()}.ent")
    first = st[0]
    ids_ca = {r.get_id()[1] for c in first for r in c if "CA" in r}
    G = build_rinq(pdb_id)
    return dict(models=len(st), chains=len(first), CA_residues=len(ids_ca),
                graph_nodes=G.number_of_nodes(), no_contact=sorted(ids_ca - set(G.nodes())))

print(pd.DataFrame({p: residue_report(p) for p in rinq_residues}).T.to_string())

Structure exists: './pdb1xy1.ent' 
Structure exists: './pdb4d5m.ent' 
Structure exists: './pdb2n08.ent' 
Structure exists: './pdb6a5j.ent' 
Structure exists: './pdb6rqs.ent' 
Structure exists: './pdb1q71.ent' 
Structure exists: './pdb2mlt.ent' 
Structure exists: './pdb1a7f.ent' 
Structure exists: './pdb1gcn.ent' 
Structure exists: './pdb2k6o.ent' 
Structure exists: './pdb1jl9.ent' 
Structure exists: './pdb1ubq.ent' 
     models chains CA_residues graph_nodes no_contact
1XY1      1      2           9           9         []
4D5M      1      4          10          10         []
2N08     20      1          11          11         []
6A5J     20      1          13          13         []
6RQS     10      1          16          16         []
1Q71     20      1          21          21         []
2MLT      1      2          26          26         []
1A7F     20      2          29          29         []
1GCN      1      1          29          29         []
2K6O      4      1          37          

In [ ]:
# ---- classical SC top-5 of each graph vs the Estrada top-5 listed in Table 2 of the RinQ paper ----
from scipy.linalg import expm

rinq_table2 = {"1A7F": [11, 6, 12, 14, 10], "1GCN": [23, 19, 20, 18, 17], "1JL9": [14, 42, 33, 31, 18],
               "1Q71": [19, 18, 7, 8, 20],  "1UBQ": [5, 4, 23, 3, 6],     "1XY1": [6, 5, 1, 3, 2],
               "2K6O": [29, 13, 10, 18, 14], "2MLT": [12, 9, 13, 8, 14],  "2N08": [6, 5, 7, 8, 4],
               "4D5M": [5, 3, 6, 8, 4],      "6A5J": [8, 9, 6, 7, 5],     "6RQS": [10, 11, 9, 8, 12]}

out = {}
for pid, ref in rinq_table2.items():
    G = build_rinq(pid)
    nodes = sorted(G.nodes())
    sc = np.diag(expm(nx.to_numpy_array(G, nodelist=nodes)))
    top = [nodes[i] for i in np.argsort(-sc, kind="stable")[:5]]
    shared = len(set(top) & set(ref))
    out[pid] = dict(ours=top, paper=ref, shared=shared, jaccard=round(shared / (10 - shared), 2), same_order=(top == ref))
print(pd.DataFrame(out).T.to_string())

Structure exists: './pdb1a7f.ent' 
Structure exists: './pdb1gcn.ent' 
Structure exists: './pdb1jl9.ent' 
Structure exists: './pdb1q71.ent' 
Structure exists: './pdb1ubq.ent' 
Structure exists: './pdb1xy1.ent' 
Structure exists: './pdb2k6o.ent' 
Structure exists: './pdb2mlt.ent' 
Structure exists: './pdb2n08.ent' 
Structure exists: './pdb4d5m.ent' 
Structure exists: './pdb6a5j.ent' 
Structure exists: './pdb6rqs.ent' 
                      ours                 paper shared jaccard same_order
1A7F   [11, 6, 12, 14, 10]   [11, 6, 12, 14, 10]      5     1.0       True
1GCN  [23, 19, 20, 18, 17]  [23, 19, 20, 18, 17]      5     1.0       True
1JL9  [14, 42, 33, 31, 18]  [14, 42, 33, 31, 18]      5     1.0       True
1Q71    [19, 18, 7, 8, 20]    [19, 18, 7, 8, 20]      5     1.0       True
1UBQ      [5, 4, 23, 3, 6]      [5, 4, 23, 3, 6]      5     1.0       True
1XY1       [6, 5, 1, 3, 2]       [6, 5, 1, 3, 2]      5     1.0       True
2K6O  [29, 13, 10, 18, 14]  [29, 13, 10, 18, 14]      5